# Week 8: Feature Importance & SHAP Values
# 第 8 週：特徵重要度與 Shapley 值

## 學習目標 Learning Objectives
- 視覺化特徵相關性 (Correlation) 熱力圖
- 理解並比較 MDI (Mean Decrease Impurity) 特徵重要度
- 掌握排列重要度 (Permutation Importance) 的原理與實作
- 實作 Drop-Column Importance 方法
- 理解 Shapley 值的博弈論背景，手動計算簡化範例
- 建立 SHAP-like 瀑布圖 (Waterfall) 與蜂群圖 (Beeswarm) 的視覺化
- 使用特徵選擇 (Feature Selection) 提升模型效能

In [ ]:
# 環境準備 Environment Setup
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.cm as cm
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error
from itertools import combinations
from math import factorial
import warnings
warnings.filterwarnings('ignore')

# 設定中文字型與繪圖風格
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

np.random.seed(42)
print('所有套件載入成功！All packages imported successfully!')

In [ ]:
# 載入 California Housing 資料集
housing = fetch_california_housing()
X = housing.data
y = housing.target
feature_names = list(housing.feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'資料集形狀 Dataset shape: {X.shape}')
print(f'特徵名稱 Feature names: {feature_names}')
print(f'目標變數 Target: Median house value (in $100k)')
print(f'訓練集大小: {X_train.shape[0]}, 測試集大小: {X_test.shape[0]}')
print(f'\n目標值統計:')
print(f'  Mean: {y.mean():.3f}, Std: {y.std():.3f}')
print(f'  Min: {y.min():.3f}, Max: {y.max():.3f}')

---
## Part 1: Correlation Analysis — 相關性分析

相關係數 (Correlation Coefficient) 衡量兩個變數之間的線性關係強度：

$$r_{xy} = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum(x_i - \bar{x})^2 \cdot \sum(y_i - \bar{y})^2}}$$

- $r = +1$：完全正相關
- $r = -1$：完全負相關
- $r = 0$：無線性相關（但可能有非線性關係）

In [ ]:
# Part 1: 相關性熱力圖
# 計算特徵間的相關係數矩陣（含目標變數）
all_data = np.column_stack([X, y])
all_names = feature_names + ['MedHouseVal']
corr_matrix = np.corrcoef(all_data.T)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

# 添加數值標註
for i in range(len(all_names)):
    for j in range(len(all_names)):
        color = 'white' if abs(corr_matrix[i, j]) > 0.6 else 'black'
        ax.text(j, i, f'{corr_matrix[i, j]:.2f}', ha='center', va='center',
                fontsize=8, color=color)

ax.set_xticks(range(len(all_names)))
ax.set_yticks(range(len(all_names)))
ax.set_xticklabels(all_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(all_names, fontsize=9)
ax.set_title('特徵相關性熱力圖 (含目標變數)\nFeature Correlation Heatmap',
             fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Correlation Coefficient', shrink=0.8)
plt.tight_layout()
plt.show()

# 與目標變數的相關性排序
target_corr = corr_matrix[-1, :-1]
sorted_idx = np.argsort(np.abs(target_corr))[::-1]
print('特徵與目標變數的相關性排序 (Feature-Target Correlation):')
for i in sorted_idx:
    print(f'  {feature_names[i]:15s}: r = {target_corr[i]:+.4f}')

In [ ]:
# 相關性分析的局限：展示非線性關係
fig, axes = plt.subplots(2, 4, figsize=(20, 9))

for i, (ax, fname) in enumerate(zip(axes.ravel(), feature_names)):
    # 取隨機子集以便視覺化
    sample_idx = np.random.choice(len(X), 500, replace=False)
    ax.scatter(X[sample_idx, i], y[sample_idx], s=5, alpha=0.4, color='#3b82f6')
    ax.set_xlabel(fname, fontsize=10)
    ax.set_ylabel('MedHouseVal' if i % 4 == 0 else '', fontsize=10)
    r = target_corr[i]
    ax.set_title(f'{fname}  (r={r:+.3f})', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.2)

fig.suptitle('各特徵與目標變數的散佈圖\nFeature vs Target Scatter Plots',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('注意：有些特徵（如 Latitude, Longitude）與目標的關係是非線性的')
print('相關係數只能捕捉線性關係，低相關性不代表不重要！')

---
## Part 2: Tree-based Feature Importance (MDI)

決策樹與集成模型在訓練過程中，會記錄每個特徵在分裂節點時所帶來的**不純度下降量**。

在 sklearn 中，`feature_importances_` 基於 **Mean Decrease Impurity (MDI)**：

$$\text{Importance}(j) = \sum_{\text{nodes using } j} \frac{N_t}{N} \cdot \Delta \text{Impurity}$$

**限制：** MDI 偏向高基數 (high-cardinality) 特徵，且基於訓練集計算。

In [ ]:
# Part 2: Tree-based Feature Importance (MDI)
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_r2 = rf.score(X_test, y_test)
print(f'Random Forest R2 Score: {rf_r2:.4f}')

# MDI Feature Importance
mdi_importances = rf.feature_importances_
mdi_sorted_idx = np.argsort(mdi_importances)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(mdi_sorted_idx)), mdi_importances[mdi_sorted_idx],
        color='#3b82f6', edgecolor='black', alpha=0.8)
ax.set_yticks(range(len(mdi_sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in mdi_sorted_idx], fontsize=10)
ax.set_xlabel('Feature Importance (MDI)', fontsize=12)
ax.set_title('Random Forest 特徵重要度 (MDI)\nMean Decrease Impurity',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# 標記數值
for i, idx in enumerate(mdi_sorted_idx):
    ax.text(mdi_importances[idx] + 0.005, i, f'{mdi_importances[idx]:.3f}',
            va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('\nMDI Top-3 特徵:')
for rank, idx in enumerate(mdi_sorted_idx[::-1][:3]):
    print(f'  {rank+1}. {feature_names[idx]}: {mdi_importances[idx]:.4f}')

---
## Part 3: Permutation Importance — 排列重要度

排列重要度的核心思路非常直覺：**如果某個特徵很重要，打亂它的值後，模型效能應該會明顯下降。**

$$\text{PI}_f = s_{\text{baseline}} - \frac{1}{K} \sum_{k=1}^{K} s_{f,k}$$

其中 $s_{\text{baseline}}$ 是原始分數，$s_{f,k}$ 是第 $k$ 次打亂特徵 $f$ 後的分數。

**優勢：** 模型不可知 (Model-Agnostic)、基於測試集、直觀易懂。

In [ ]:
# Part 3: Permutation Importance
perm_result = permutation_importance(rf, X_test, y_test,
                                      n_repeats=30, random_state=42, n_jobs=-1)

perm_importances = perm_result.importances_mean
perm_std = perm_result.importances_std
perm_sorted_idx = np.argsort(perm_importances)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 左圖：Permutation Importance（含誤差棒）
axes[0].barh(range(len(perm_sorted_idx)), perm_importances[perm_sorted_idx],
             xerr=perm_std[perm_sorted_idx], color='#10b981', edgecolor='black',
             alpha=0.8, capsize=3)
axes[0].set_yticks(range(len(perm_sorted_idx)))
axes[0].set_yticklabels([feature_names[i] for i in perm_sorted_idx], fontsize=10)
axes[0].set_xlabel('Decrease in R2 Score', fontsize=12)
axes[0].set_title('排列重要度\nPermutation Importance', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# 右圖：MDI vs Permutation 排名比較
mdi_ranks = np.argsort(np.argsort(mdi_importances)[::-1]) + 1
perm_ranks = np.argsort(np.argsort(perm_importances)[::-1]) + 1

x_pos = np.arange(len(feature_names))
width = 0.35
axes[1].bar(x_pos - width/2, mdi_ranks, width, color='#3b82f6', alpha=0.7,
            edgecolor='black', label='MDI Rank')
axes[1].bar(x_pos + width/2, perm_ranks, width, color='#10b981', alpha=0.7,
            edgecolor='black', label='Permutation Rank')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(feature_names, rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('Rank (1=most important)', fontsize=11)
axes[1].set_title('MDI vs Permutation 排名比較\nRank Comparison', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('觀察：MDI 和 Permutation Importance 的排名可能不完全一致')
print('Permutation Importance 基於測試集，更能反映真實的泛化重要度')

---
## Part 4: Drop-Column Importance

Drop-Column Importance 是最直覺的方法：
1. 訓練完整模型，得到基準分數
2. 逐一移除每個特徵，重新訓練模型
3. 比較分數差異

**優點：** 直覺簡單  
**缺點：** 計算成本高（需要重新訓練 $p+1$ 次），且會改變模型結構

In [ ]:
# Part 4: Drop-Column Importance
# 使用較少的 n_estimators 以加速
rf_fast = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_fast.fit(X_train, y_train)
baseline_score = rf_fast.score(X_test, y_test)
print(f'Baseline R2 (all features): {baseline_score:.4f}')

drop_importances = []
for i in range(X.shape[1]):
    # 移除第 i 個特徵
    X_train_drop = np.delete(X_train, i, axis=1)
    X_test_drop = np.delete(X_test, i, axis=1)

    rf_drop = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_drop.fit(X_train_drop, y_train)
    drop_score = rf_drop.score(X_test_drop, y_test)
    drop_importances.append(baseline_score - drop_score)
    print(f'  Drop {feature_names[i]:15s}: R2 = {drop_score:.4f}  '
          f'(delta = {baseline_score - drop_score:+.4f})')

drop_importances = np.array(drop_importances)
drop_sorted_idx = np.argsort(drop_importances)

fig, ax = plt.subplots(figsize=(10, 6))
colors_drop = ['#dc2626' if v < 0 else '#f59e0b' for v in drop_importances[drop_sorted_idx]]
ax.barh(range(len(drop_sorted_idx)), drop_importances[drop_sorted_idx],
        color=colors_drop, edgecolor='black', alpha=0.8)
ax.set_yticks(range(len(drop_sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in drop_sorted_idx], fontsize=10)
ax.set_xlabel('Decrease in R2 Score (Baseline - Drop)', fontsize=12)
ax.set_title('Drop-Column Importance\n移除單一特徵後的 R2 下降量',
             fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=1)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('\n注意：負值表示移除該特徵後模型效能反而提升（可能是雜訊特徵）')

---
## Part 5: SHAP Concept — Shapley 值的理論

SHAP 的核心建立在合作博弈論中的 **Shapley 值** 之上（Lloyd Shapley, 1953；2012 諾貝爾經濟學獎）。

**博弈論類比：**
- 玩家 (Players) = 特徵 (Features)
- 聯盟 (Coalition) = 特徵的子集合
- 報酬 (Payout) = 模型預測值

$$\phi_i = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|! \cdot (|N| - |S| - 1)!}{|N|!} \left[ v(S \cup \{i\}) - v(S) \right]$$

**四大公理 (Four Axioms):**
1. **效率性 (Efficiency):** 所有 SHAP 值之和 = 預測值 - 基準值
2. **對稱性 (Symmetry):** 相同邊際貢獻的特徵獲得相同 SHAP 值
3. **虛擬性 (Null Player):** 不影響預測的特徵，SHAP 值為 0
4. **可加性 (Additivity):** 多個遊戲的 SHAP 值等於個別 SHAP 值的和

下面我們用 3 個特徵的小範例，手動計算 Shapley 值。

In [ ]:
# Part 5: 手動計算 Shapley 值 — 3 個特徵的簡化範例
# 建立一個小型資料集和模型
np.random.seed(42)
n_samples = 500
x1 = np.random.randn(n_samples)
x2 = np.random.randn(n_samples)
x3 = np.random.randn(n_samples)
# 真實函數: y = 3*x1 + 2*x2 + 0.5*x3 + noise
y_small = 3 * x1 + 2 * x2 + 0.5 * x3 + 0.3 * np.random.randn(n_samples)
X_small = np.column_stack([x1, x2, x3])
small_features = ['x1', 'x2', 'x3']

# 訓練模型
model_small = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
model_small.fit(X_small, y_small)

# 選取一個樣本來解釋
sample_idx = 0
x_explain = X_small[sample_idx:sample_idx+1]
pred_explain = model_small.predict(x_explain)[0]
base_value = y_small.mean()  # 基準值 (平均預測)

print(f'要解釋的樣本: x1={x_explain[0,0]:.3f}, x2={x_explain[0,1]:.3f}, x3={x_explain[0,2]:.3f}')
print(f'模型預測值 f(x): {pred_explain:.3f}')
print(f'基準值 (平均預測): {base_value:.3f}')
print(f'需要分配的貢獻: {pred_explain - base_value:.3f}')
print()

# 手動列舉所有聯盟計算 Shapley 值
N = [0, 1, 2]  # 特徵索引
n_features_small = len(N)

def compute_v(subset, x_sample, model, X_bg):
    """計算聯盟 subset 的價值函數 v(S)
    使用背景資料的邊際化 (marginalization) 來估計"""
    if len(subset) == 0:
        return model.predict(X_bg).mean()
    # 建構替換後的資料：subset 中的特徵用 x_sample，其餘用背景資料
    X_modified = X_bg.copy()
    for j in subset:
        X_modified[:, j] = x_sample[0, j]
    return model.predict(X_modified).mean()

# 使用部分背景資料
X_bg = X_small[:100]

shapley_values = []
print('=== Shapley Value Calculation ===')
for i in range(n_features_small):
    phi_i = 0.0
    others = [j for j in N if j != i]
    print(f'\nFeature {small_features[i]} (index={i}):')

    # 列舉所有不含 i 的子集 S
    for size in range(len(others) + 1):
        for S in combinations(others, size):
            S_list = list(S)
            S_with_i = S_list + [i]

            v_with = compute_v(S_with_i, x_explain, model_small, X_bg)
            v_without = compute_v(S_list, x_explain, model_small, X_bg)
            marginal = v_with - v_without

            # Shapley 權重
            weight = (factorial(len(S_list)) * factorial(n_features_small - len(S_list) - 1)) / factorial(n_features_small)
            contribution = weight * marginal
            phi_i += contribution

            print(f'  S={set(S_list)}, v(S)={v_without:.3f}, '
                  f'v(S+{{{i}}})={v_with:.3f}, '
                  f'marginal={marginal:+.3f}, weight={weight:.3f}')

    shapley_values.append(phi_i)
    print(f'  => phi_{small_features[i]} = {phi_i:+.4f}')

print(f'\n--- Summary ---')
base_v = compute_v([], x_explain, model_small, X_bg)
full_v = compute_v(list(range(n_features_small)), x_explain, model_small, X_bg)
print(f'Base value:   {base_v:.4f}')
print(f'Sum of SHAP:  {sum(shapley_values):+.4f}')
print(f'Base + SHAP:  {base_v + sum(shapley_values):.4f}')
print(f'Actual pred:  {full_v:.4f}')
print('\nEfficiency axiom: Base + sum(SHAP) should approximately equal prediction')

---
## Part 6: SHAP-like Waterfall Visualization

瀑布圖 (Waterfall Plot) 展示單一樣本的預測如何由基準值推移到最終預測值：

$$f(x) = \phi_0 + \phi_1 + \phi_2 + \cdots + \phi_M$$

其中 $\phi_0$ 是基準值 (base value)，$\phi_i$ 是特徵 $i$ 的 SHAP 值。

In [ ]:
# Part 6: SHAP-like Waterfall / Force Plot
base_val = compute_v([], x_explain, model_small, X_bg)
shap_vals = np.array(shapley_values)

# 按絕對值排序
abs_sorted = np.argsort(np.abs(shap_vals))[::-1]

fig, axes = plt.subplots(2, 1, figsize=(12, 9))

# 上圖：水平瀑布圖 (Waterfall)
ax = axes[0]
cumulative = base_val
positions = []
widths = []
colors_wf = []
labels_wf = []

for idx in abs_sorted:
    val = shap_vals[idx]
    positions.append(cumulative)
    widths.append(val)
    colors_wf.append('#dc2626' if val > 0 else '#2563eb')
    labels_wf.append(f'{small_features[idx]}={x_explain[0, idx]:.2f}')
    cumulative += val

y_positions = range(len(abs_sorted))
for i, (pos, w, c, label) in enumerate(zip(positions, widths, colors_wf, labels_wf)):
    ax.barh(i, w, left=pos, color=c, edgecolor='black', alpha=0.8, height=0.6)
    sign = '+' if w > 0 else ''
    ax.text(pos + w/2, i, f'{sign}{w:.3f}', ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')

ax.set_yticks(list(y_positions))
ax.set_yticklabels(labels_wf, fontsize=11)
ax.axvline(x=base_val, color='gray', linestyle='--', linewidth=1.5, label=f'Base={base_val:.3f}')
ax.axvline(x=cumulative, color='black', linestyle='-', linewidth=2, label=f'Pred={cumulative:.3f}')
ax.set_xlabel('Prediction Value', fontsize=12)
ax.set_title('SHAP Waterfall Plot (Single Prediction)\nSHAP 瀑布圖 — 單一預測的分解',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='x')

# 下圖：Force-style 水平堆積圖
ax2 = axes[1]
pos_vals = [(small_features[i], shap_vals[i]) for i in range(n_features_small) if shap_vals[i] > 0]
neg_vals = [(small_features[i], shap_vals[i]) for i in range(n_features_small) if shap_vals[i] <= 0]

pos_vals.sort(key=lambda x: x[1], reverse=True)
neg_vals.sort(key=lambda x: x[1])

# 繪製正向貢獻（從 base 向右推）
left = base_val
for name, val in pos_vals:
    ax2.barh(0, val, left=left, color='#dc2626', edgecolor='white',
             alpha=0.8, height=0.5)
    ax2.text(left + val/2, 0, f'{name}\n+{val:.3f}', ha='center', va='center',
             fontsize=10, fontweight='bold', color='white')
    left += val

# 繪製負向貢獻（從 prediction 向左推）
right = cumulative
for name, val in neg_vals:
    ax2.barh(0, abs(val), left=right, color='#2563eb', edgecolor='white',
             alpha=0.8, height=0.5)
    ax2.text(right + abs(val)/2, 0, f'{name}\n{val:.3f}', ha='center', va='center',
             fontsize=10, fontweight='bold', color='white')
    right += abs(val)

ax2.axvline(x=base_val, color='gray', linestyle='--', linewidth=2)
ax2.text(base_val, -0.5, f'Base\n{base_val:.3f}', ha='center', fontsize=10)
ax2.axvline(x=cumulative, color='black', linestyle='-', linewidth=2)
ax2.text(cumulative, -0.5, f'Pred\n{cumulative:.3f}', ha='center', fontsize=10)

ax2.set_yticks([])
ax2.set_xlabel('Prediction Value', fontsize=12)
ax2.set_title('SHAP Force Plot (Single Prediction)\nSHAP 力圖',
             fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print('紅色 = 推高預測的特徵，藍色 = 壓低預測的特徵')
print(f'Base ({base_val:.3f}) + SHAP contributions = Prediction ({cumulative:.3f})')

---
## Part 7: SHAP-like Beeswarm Plot

蜂群圖 (Beeswarm Plot / Summary Plot) 是 SHAP 最常用的**全域解釋**圖表。

- **Y 軸**: 特徵名稱（依重要度排序）
- **X 軸**: SHAP 值（正值 → 推高預測，負值 → 壓低預測）
- **顏色**: 特徵值的高低（紅 = 高值，藍 = 低值）
- **每個點**: 一個樣本

我們用簡化方法（mean-replacement approximation）來近似全部樣本的 SHAP 值。

In [ ]:
# Part 7: SHAP-like Beeswarm Plot
# 使用 California Housing 資料集
# 近似 SHAP：對每個特徵，用訓練集均值替換後計算預測差

n_display = 200
X_subset = X_test[:n_display]
y_pred_subset = rf.predict(X_subset)

shap_approx = np.zeros((n_display, len(feature_names)))

for j in range(len(feature_names)):
    X_permuted = X_subset.copy()
    X_permuted[:, j] = X_train[:, j].mean()
    y_pred_permuted = rf.predict(X_permuted)
    shap_approx[:, j] = y_pred_subset - y_pred_permuted

# 計算全域重要度（|SHAP| 均值）用於排序
global_importance = np.abs(shap_approx).mean(axis=0)
sorted_features = np.argsort(global_importance)[::-1]

# 繪製 Beeswarm Plot
fig, ax = plt.subplots(figsize=(12, 7))

for rank, feat_idx in enumerate(sorted_features):
    shap_col = shap_approx[:, feat_idx]
    feat_col = X_subset[:, feat_idx]

    feat_min, feat_max = feat_col.min(), feat_col.max()
    if feat_max > feat_min:
        feat_norm = (feat_col - feat_min) / (feat_max - feat_min)
    else:
        feat_norm = np.zeros_like(feat_col)

    jitter = np.random.normal(0, 0.12, size=n_display)

    scatter = ax.scatter(shap_col, rank + jitter, c=feat_norm,
                         cmap='coolwarm', s=8, alpha=0.6, edgecolors='none')

ax.set_yticks(range(len(sorted_features)))
ax.set_yticklabels([feature_names[i] for i in sorted_features], fontsize=10)
ax.set_xlabel('SHAP Value (impact on prediction)', fontsize=12)
ax.set_title('SHAP-like Beeswarm Plot (California Housing)\n'
             'SHAP 蜂群圖 — 全域特徵重要度',
             fontsize=14, fontweight='bold')
ax.axvline(x=0, color='gray', linestyle='-', linewidth=1)
ax.grid(True, alpha=0.2, axis='x')
ax.invert_yaxis()

cbar = plt.colorbar(scatter, ax=ax, shrink=0.6)
cbar.set_label('Feature Value (Low -> High)', fontsize=10)
cbar.set_ticks([0, 0.5, 1])
cbar.set_ticklabels(['Low', 'Mid', 'High'])

plt.tight_layout()
plt.show()

print('解讀方式：')
print('- 紅點在右側 -> 高特徵值推高預測（正相關）')
print('- 藍點在右側 -> 低特徵值推高預測（負相關）')
print('- 點的分散程度 -> 該特徵的影響力大小')

In [ ]:
# SHAP-like Dependence Plot — 觀察單一特徵值與 SHAP 值的關係
top4 = sorted_features[:4]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feat_idx in zip(axes.ravel(), top4):
    shap_col = shap_approx[:, feat_idx]
    feat_col = X_subset[:, feat_idx]

    # 選擇最強交互特徵（與 SHAP 殘差相關性最高的其他特徵）
    other_feats = [f for f in range(len(feature_names)) if f != feat_idx]
    interact_corr = []
    for f in other_feats:
        residual = shap_col - np.polyval(np.polyfit(feat_col, shap_col, 1), feat_col)
        r = abs(np.corrcoef(residual, X_subset[:, f])[0, 1])
        interact_corr.append(r if not np.isnan(r) else 0)
    best_interact = other_feats[np.argmax(interact_corr)]

    interact_col = X_subset[:, best_interact]
    interact_norm = (interact_col - interact_col.min()) / (interact_col.max() - interact_col.min() + 1e-10)

    sc = ax.scatter(feat_col, shap_col, c=interact_norm, cmap='coolwarm',
                     s=15, alpha=0.7, edgecolors='none')
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.8)

    # 趨勢線
    z = np.polyfit(feat_col, shap_col, 2)
    x_line = np.linspace(feat_col.min(), feat_col.max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), 'k--', linewidth=1.5, alpha=0.5)

    ax.set_xlabel(f'{feature_names[feat_idx]}', fontsize=11)
    ax.set_ylabel('SHAP Value', fontsize=11)
    ax.set_title(f'{feature_names[feat_idx]} (color={feature_names[best_interact]})',
                fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.2)

fig.suptitle('SHAP-like Dependence Plots (Top-4 Features)\n'
             'SHAP 依賴圖 — 特徵值與 SHAP 值的關係',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('解讀方式：')
print('- X 軸 = 特徵的實際值，Y 軸 = 該特徵的 SHAP 值')
print('- 趨勢線顯示特徵值與 SHAP 的非線性關係')
print('- 顏色 = 交互特徵的值（自動選擇最強交互特徵）')
print('- 若同一 X 值的點因顏色不同而 Y 值不同 -> 存在特徵交互作用')

In [ ]:
# 三種重要度方法的全面比較
fig, ax = plt.subplots(figsize=(14, 7))

# 正規化到 [0, 1] 範圍以便比較
def normalize_arr(arr):
    mn, mx = arr.min(), arr.max()
    if mx > mn:
        return (arr - mn) / (mx - mn)
    return np.zeros_like(arr)

mdi_norm = normalize_arr(mdi_importances)
perm_norm = normalize_arr(perm_importances)
drop_norm = normalize_arr(drop_importances)

x_pos = np.arange(len(feature_names))
width = 0.25

ax.bar(x_pos - width, mdi_norm, width, color='#3b82f6', alpha=0.8,
       edgecolor='black', label='MDI (tree-based)')
ax.bar(x_pos, perm_norm, width, color='#10b981', alpha=0.8,
       edgecolor='black', label='Permutation')
ax.bar(x_pos + width, drop_norm, width, color='#f59e0b', alpha=0.8,
       edgecolor='black', label='Drop-Column')

ax.set_xticks(x_pos)
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Normalized Importance [0-1]', fontsize=12)
ax.set_title('三種特徵重要度方法的全面比較 (正規化)\n'
             'Comprehensive Comparison of 3 Feature Importance Methods',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('觀察重點：')
print('- 三種方法對最重要的 2-3 個特徵通常有共識')
print('- 對中等重要度的特徵排名可能不同')
print('- MDI 可能高估訓練中頻繁使用但泛化不重要的特徵')
print('- Permutation 和 Drop-Column 基於測試集效能，更為可靠')

---
## Part 8: Feature Selection — 特徵選擇

使用特徵重要度來進行特徵選擇：選取 top-k 個特徵，比較模型效能。

理論上，移除不重要的特徵可以：
- 減少過擬合風險
- 降低訓練/預測時間
- 提升模型可解釋性

In [ ]:
# Part 8: Feature Selection — 選取 top-k 特徵比較模型效能
perm_ranking = np.argsort(perm_importances)[::-1]  # 從高到低

k_values = range(1, len(feature_names) + 1)
cv_scores_k = []
test_scores_k = []

for k in k_values:
    top_k_features = perm_ranking[:k]
    X_train_k = X_train[:, top_k_features]
    X_test_k = X_test[:, top_k_features]

    rf_k = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_k.fit(X_train_k, y_train)

    # 5-fold CV
    X_k = X[:, top_k_features]
    cv = cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
                         X_k, y, cv=5, scoring='r2')
    cv_scores_k.append(cv.mean())
    test_scores_k.append(rf_k.score(X_test_k, y_test))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(list(k_values), cv_scores_k, 'o-', color='#2563eb', linewidth=2,
        markersize=8, label='5-Fold CV R2')
ax.plot(list(k_values), test_scores_k, 's--', color='#dc2626', linewidth=2,
        markersize=8, label='Test R2', alpha=0.7)

best_k_cv = np.argmax(cv_scores_k) + 1
ax.axvline(x=best_k_cv, color='green', linestyle=':', alpha=0.7,
           label=f'Best k={best_k_cv} (CV)')

ax.set_xlabel('Number of Top Features (k)', fontsize=12)
ax.set_ylabel('R2 Score', fontsize=12)
ax.set_title('特徵選擇：Top-k 特徵 vs 模型效能\nFeature Selection: Top-k Features vs Performance',
             fontsize=14, fontweight='bold')
ax.set_xticks(list(k_values))
ax.set_xticklabels([feature_names[perm_ranking[i]] if i < len(feature_names) else ''
                     for i in range(len(k_values))],
                    rotation=45, ha='right', fontsize=8)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'最佳特徵數量 (CV): k={best_k_cv}')
print(f'使用 {best_k_cv} 個特徵的 CV R2: {cv_scores_k[best_k_cv-1]:.4f}')
print(f'使用全部特徵的 CV R2: {cv_scores_k[-1]:.4f}')
print(f'\nTop-{best_k_cv} features by Permutation Importance:')
for i in range(best_k_cv):
    print(f'  {i+1}. {feature_names[perm_ranking[i]]}')

---
## Exercises — 練習題

### 練習 1：比較 MDI vs Permutation Importance 對相關特徵的處理

建立一個人工資料集，其中有兩個高度相關的特徵 (correlation > 0.95)，
觀察 MDI 和 Permutation Importance 如何分配它們的重要度。

**提示：** 可以用 `x2 = x1 + 0.05 * noise` 來建立高度相關的特徵。

In [ ]:
# 練習 1：MDI vs Permutation Importance for correlated features
# TODO: 建立人工資料集 (x1, x2=x1+noise, x3=random, y=3*x1+noise)
# TODO: 訓練 RandomForest 並取得 MDI feature_importances_
# TODO: 計算 permutation_importance
# TODO: 繪製比較圖
# TODO: 觀察兩個方法對高相關特徵 (x1, x2) 重要度分配的差異


### 練習 2：實作遞迴特徵消除 (Recursive Feature Elimination)

使用 California Housing 資料集，用 `sklearn.feature_selection.RFE` 搭配 RandomForest
進行特徵選擇，找出最佳特徵子集。

**提示：** `RFE(estimator, n_features_to_select=k)`

In [ ]:
# 練習 2：Recursive Feature Elimination (RFE)
# from sklearn.feature_selection import RFE
# TODO: 使用 RFE 搭配 RandomForestRegressor
# TODO: 嘗試不同的 n_features_to_select (1 到 8)
# TODO: 對每個 k 計算 R2 score
# TODO: 比較 RFE 選擇的特徵順序與 Permutation Importance 的排名


### 練習 3：分析不同預測範圍的關鍵特徵

將 California Housing 的預測結果分為「低價」(pred < 1.5)、「中價」(1.5-3.5)、「高價」(pred > 3.5)，
對每個價位區間分別計算 Permutation Importance，觀察不同價位段的關鍵特徵是否不同。

**提示：** 將測試集依預測值分組，分別計算 `permutation_importance`。

In [ ]:
# 練習 3：不同預測範圍的特徵重要度
# TODO: 用 rf 預測測試集，依預測值分為三組
# TODO: 對每組分別計算 permutation_importance
# TODO: 繪製三組的特徵重要度比較圖（分組 bar chart）
# TODO: 分析哪些特徵在不同價位段的重要度差異最大


---
## Summary — 本週重點回顧

### 關鍵概念 Key Takeaways

1. **相關性分析 (Correlation)** 是最基礎的特徵分析方法，但只能捕捉線性關係
2. **MDI (Mean Decrease Impurity)** 是樹模型內建的特徵重要度，計算快速但可能偏向高基數特徵
3. **Permutation Importance** 是模型不可知的方法，基於測試集，更能反映泛化重要度
4. **Drop-Column Importance** 最直覺但計算成本高（需重新訓練模型）
5. **Shapley 值** 來自合作博弈論，是唯一滿足四大公理（效率性、對稱性、虛擬性、可加性）的公平分配方式
6. **SHAP 圖表** 包含瀑布圖（局部解釋）、蜂群圖（全域解釋）和依賴圖（特徵交互），可以直觀展示特徵貢獻
7. **特徵選擇** 可以移除不重要特徵，減少過擬合風險並提升模型效能

### 下週預告 Next Week Preview
**第 9 週：特徵工程與資料前處理管線 (Feature Engineering & Preprocessing Pipeline)** — 
學習類別編碼 (One-Hot, Ordinal)、特徵縮放 (Standard, MinMax, Robust)、缺失值處理、
sklearn Pipeline 與 ColumnTransformer 的完整工作流程。